# SI/MA neural coding: Glutamatergic vs. GABAergic groups

This notebook reproduces the polar / diagonal significance analysis of the
reference call

```python
run_model_plot(
    ds=ds,
    plot_methods=["polar", "diag"],
    models=["qlearn_reward_vs_chosenQ_g0"],
    regions=[["SI", "MA"]],
    time_window="0.3_2_-1_0",
    ...
)
```

but **splits the SI/MA (ventral pallidum) units into two putative cell-type
groups** using waveform morphology, then plots the polar figure **separately**
for each group so the neural coding can be compared.

**Strategy**
1. Load the per-unit correlation results (`ds`).
2. Load the waveform-feature table and the opto-tagged (Glutamatergic) units.
3. Cluster the SI/MA population into two groups on waveform features. The
   cluster that captures the majority of the opto-tagged units is labelled
   **Glutamatergic**; the other is labelled **GABAergic**.
4. Map those group labels back onto `ds` (via `session_name` + `unit_index`),
   split `ds`, and draw the polar plot for each group.

## 1. Environment setup & imports

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
import re
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from general_utils import load_temporary_data
from waveform_clustering import (
    FEATURE_COLS,
    load_features_dataset,
    normalize_waveform,
    clean_waveform_mask,
    cluster_waveforms,
)
from ephys_behavior_visualization import (
    plot_angle_fraction_polar,
    plot_diagonal_significance,
)

%matplotlib inline
print("Imports ready.")

## 2. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Correlation results (per-unit regression coefficients / p-values) -------
ZARR_PATH = "/root/capsule/scratch/correlation_results/sig_dir_all_sessions.zarr"

# --- Waveform features + opto-tagging artifacts ------------------------------
DATASET_CSV = Path("/root/capsule/scratch/waveform_clustering/all_sessions_waveform_features.csv")
OPTO_DIR = Path("/root/capsule/scratch/opto_tagging_Anna")

# --- Region + analysis window ------------------------------------------------
TARGET_REGIONS = ["SI", "MA"]          # ventral pallidum
TIME_WINDOW = "0.3_2_-1_0"             # same window as the reference polar plot

# --- Model columns (qlearn_reward_vs_chosenQ_g0) -----------------------------
MODEL_COLS = {
    "col_x": "simple_LR-QLearning_L2F1_softmax-reward-g0-s0-d0-coef",
    "col_y": "simple_LR-QLearning_L2F1_softmax-chosenQ-g0-s0-d0-coef",
    "col_pval_x": "simple_LR-QLearning_L2F1_softmax-reward-g0-s0-d0-pval",
    "col_pval_y": "simple_LR-QLearning_L2F1_softmax-chosenQ-g0-s0-d0-pval",
}

# --- Polar plot options (match the reference call) ---------------------------
POLAR_KW = dict(
    include="all",
    angle_bin_deg=15,
    normalize="selected",
    start_angle_deg=0,
    show_reference_diagonals=True,
)

# --- Opto-tagging thresholds (same as tagged_vs_region_waveforms.ipynb) -------
RED_MIN_SIG_PULSES = 4
BLUE_MIN_SIG_PULSES = 5
MAX_JITTER = 0.01
MAX_ISI = 0.5
TAG_TYPES = ["external_blue"]          # None => all tag types

# --- Noise filtering for waveforms -------------------------------------------
FILTER_NOISE = True
NOISE_MAX_PEAKS = 4
NOISE_MAX_ZERO_CROSSINGS = 8
NOISE_MAX_LATE_ENERGY = 0.6
NOISE_CORE_MS = 0.7

# --- Which waveform features drive the two-group split -----------------------
# Default: all FEATURE_COLS. The classic separators are trough_to_peak_ms,
# half_width_ms and peak_trough_ratio; restrict here for a cleaner split.
GROUP_FEATURE_COLS = list(FEATURE_COLS)

print("Config set. Window:", TIME_WINDOW, "| Regions:", TARGET_REGIONS)

## 3. Load the correlation results (`ds`)

In [ ]:
# One row per unit per time window (long form). Units are keyed by
# (session_name, unit_index); the region column is `brain_region`.
ds = load_temporary_data(ZARR_PATH)
print("ds rows:", len(ds))
print("Key columns present:",
      [c for c in ["brain_region", "time_window", "session_name", "unit_index"] if c in ds.columns])
ds.head()

## 4. Load the waveform features and build the SI/MA population mask

In [ ]:
# Reload the flat waveform/feature CSV into a feature table + waveform matrix.
loaded = load_features_dataset(DATASET_CSV)
features = loaded["features"]
waveforms = loaded["waveforms"]
time_ms = loaded["time_ms"]

# Amplitude-normalize each stored waveform (used by the noise filter + plots).
norm_waveforms = (
    np.vstack([normalize_waveform(w) for w in waveforms]) if len(waveforms) else waveforms
)

# QC mask.
qc_mask = features["qc_pass"].astype(bool).values

# Noise / artifact filter (drop oscillatory, non-spike waveforms).
if FILTER_NOISE:
    _cm = clean_waveform_mask(
        norm_waveforms, time_ms,
        max_peaks=NOISE_MAX_PEAKS,
        max_zero_crossings=NOISE_MAX_ZERO_CROSSINGS,
        max_late_energy=NOISE_MAX_LATE_ENERGY,
        core_ms=NOISE_CORE_MS,
    )
    clean_mask = np.asarray(_cm[0] if isinstance(_cm, tuple) else _cm, dtype=bool)
else:
    clean_mask = np.ones(len(features), dtype=bool)

# SI/MA population (QC + non-noise).
region_mask = features["region"].isin(TARGET_REGIONS).values & qc_mask & clean_mask
print(f"Total units: {len(features)} | QC-pass: {int(qc_mask.sum())} | non-noise: {int(clean_mask.sum())}")
print(f"SI/MA units (QC + non-noise): {int(region_mask.sum())}")
print(features.loc[region_mask, "region"].value_counts())

## 5. Identify the opto-tagged (Glutamatergic) units

Same tagging query as `tagged_vs_region_waveforms.ipynb`: read each
`*_laser_response_metrics.csv`, keep units with enough significant pulses and
low jitter, and match them into the feature table by `session_name` +
`unit_index`.

In [ ]:
SESSION_CORE_RE = re.compile(r"(\d+_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")


def tagged_from_metrics(metrics, trial_type, min_sig_pulses, max_jitter, max_isi):
    """Rows of `metrics` passing the tagging criteria for one trial type."""
    q = []
    col_pulses = f"{trial_type}_train_max_num_sig_pulses"
    col_jitter = f"{trial_type}_train_best_mean_jitter"
    if col_pulses in metrics.columns:
        q.append(f"{col_pulses} >= {min_sig_pulses}")
    if col_jitter in metrics.columns:
        q.append(f"{col_jitter} < {max_jitter}")
    if "pre_stim_isi_ratio" in metrics.columns:
        q.append(f"pre_stim_isi_ratio < {max_isi}")
    if not q:
        return metrics.iloc[0:0]
    return metrics.query(" and ".join(q))


def load_tagged_units(opto_dir, red_min_sig_pulses, blue_min_sig_pulses, max_jitter, max_isi):
    """Collect tagged (session_name, unit_index, tag_type) from metric CSVs."""
    rows = []
    csvs = sorted(Path(opto_dir).glob("*_laser_response_metrics.csv"))
    print(f"Found {len(csvs)} metric CSV(s) in {opto_dir}")
    for csv in csvs:
        m = SESSION_CORE_RE.search(csv.name)
        if not m:
            continue
        session_core = m.group(1)
        metrics = pd.read_csv(csv)
        if "unit_id" not in metrics.columns:
            continue
        trial_types = sorted({
            c[: -len("_train_max_num_sig_pulses")]
            for c in metrics.columns if c.endswith("_train_max_num_sig_pulses")
        })
        red_types = [t for t in trial_types if "red" in t]
        blue_types = [t for t in trial_types if "blue" in t]
        red_idx = set()
        for tt in red_types:
            tg = tagged_from_metrics(metrics, tt, red_min_sig_pulses, max_jitter, max_isi)
            red_idx.update(tg.index.tolist())
            for uid in tg["unit_id"].astype(int):
                rows.append((session_core, int(uid), tt))
        for tt in blue_types:
            tg = tagged_from_metrics(metrics, tt, blue_min_sig_pulses, max_jitter, max_isi)
            tg = tg[~tg.index.isin(red_idx)]
            for uid in tg["unit_id"].astype(int):
                rows.append((session_core, int(uid), tt))
    tagged = pd.DataFrame(rows, columns=["session_name", "unit_index", "tag_type"])
    return tagged.drop_duplicates(subset=["session_name", "unit_index"]).reset_index(drop=True)


tagged_df = load_tagged_units(
    OPTO_DIR,
    red_min_sig_pulses=RED_MIN_SIG_PULSES,
    blue_min_sig_pulses=BLUE_MIN_SIG_PULSES,
    max_jitter=MAX_JITTER,
    max_isi=MAX_ISI,
)

if TAG_TYPES is not None:
    tagged_sel = tagged_df[tagged_df["tag_type"].isin(TAG_TYPES)].copy()
else:
    tagged_sel = tagged_df.copy()


def make_key(df):
    return df["session_name"].astype(str) + "|" + df["unit_index"].astype(int).astype(str)


feat_key = make_key(features)
tagged_key = set(make_key(tagged_sel))
tagged_mask = feat_key.isin(tagged_key).values & qc_mask & clean_mask
print(f"\nTagged (Glutamatergic) units matched: {int(tagged_mask.sum())}")
print(f"  of those in SI/MA: {int((tagged_mask & region_mask).sum())}")

## 6. Split SI/MA into two waveform groups, anchored by the tagged units

A 2-component clustering is fit on the SI/MA waveform features. Whichever
cluster captures the majority of the opto-tagged (Glutamatergic) units is
labelled **Glutamatergic**; the other is labelled **GABAergic**. If no tagged
unit lands in SI/MA, we fall back to the classic broad-spiking heuristic
(larger `trough_to_peak_ms` = Glutamatergic).

In [ ]:
sima_feat = features.loc[region_mask].copy().reset_index(drop=True)
sima_tagged = tagged_mask[region_mask]            # tagged status, aligned to sima_feat rows

# 2-group clustering on waveform morphology (adds cluster_kmeans / cluster_gmm).
clust = cluster_waveforms(sima_feat, n_clusters=2, feature_cols=GROUP_FEATURE_COLS)
labels = sima_feat["cluster_gmm"].to_numpy()

# Anchor: which cluster do the tagged units mostly fall into?
if sima_tagged.sum() > 0:
    glut_cluster = int(np.bincount(labels[sima_tagged]).argmax())
    anchor = f"tagged units (n={int(sima_tagged.sum())})"
else:
    glut_cluster = int(sima_feat.groupby("cluster_gmm")["trough_to_peak_ms"].mean().idxmax())
    anchor = "broad-spiking heuristic (no tagged units in SI/MA)"

other_cluster = [c for c in np.unique(labels) if c != glut_cluster][0]
group_names = {glut_cluster: "Glutamatergic", other_cluster: "GABAergic"}
sima_feat["wf_group"] = sima_feat["cluster_gmm"].map(group_names)

print(f"Anchored Glutamatergic group using: {anchor}")
print(sima_feat["wf_group"].value_counts())

if sima_tagged.sum() > 0:
    frac = (sima_feat.loc[sima_tagged, "wf_group"] == "Glutamatergic").mean()
    print(f"Tagged units assigned to Glutamatergic group: {frac * 100:.1f}%")

# Per-group feature means (sanity check of the split).
sima_feat.groupby("wf_group")[GROUP_FEATURE_COLS].mean()

In [ ]:
# Visualize the two groups in waveform feature space, with tagged units marked.
fig, ax = plt.subplots(figsize=(7.5, 6))
colors = {"Glutamatergic": "tab:blue", "GABAergic": "tab:orange"}
for g, c in colors.items():
    m = (sima_feat["wf_group"] == g).to_numpy()
    ax.scatter(sima_feat.loc[m, "trough_to_peak_ms"],
               sima_feat.loc[m, "half_width_ms"],
               s=25, c=c, alpha=0.6, edgecolor="none",
               label=f"{g} (n={int(m.sum())})")
ax.scatter(sima_feat.loc[sima_tagged, "trough_to_peak_ms"],
           sima_feat.loc[sima_tagged, "half_width_ms"],
           s=120, marker="*", facecolor="none", edgecolor="crimson", linewidth=1.2,
           label=f"Tagged (n={int(sima_tagged.sum())})", zorder=5)
ax.set_xlabel("Trough-to-peak duration (ms)")
ax.set_ylabel("Half-width (ms)")
ax.set_title("SI/MA units split into two waveform groups")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Map the group labels onto `ds` and split it

`plot_angle_fraction_polar` filters `ds` only by region + time window, so we
pre-split `ds` into a Glutamatergic subset and a GABAergic subset using the
`(session_name, unit_index)` group map.

In [ ]:
group_map = {
    f"{s}|{int(u)}": g
    for s, u, g in zip(sima_feat["session_name"], sima_feat["unit_index"], sima_feat["wf_group"])
}

ds_sima = ds[(ds["brain_region"].isin(TARGET_REGIONS)) & (ds["time_window"] == TIME_WINDOW)].copy()
ds_key = ds_sima["session_name"].astype(str) + "|" + ds_sima["unit_index"].astype(int).astype(str)
ds_sima["wf_group"] = ds_key.map(group_map)

n_unmapped = int(ds_sima["wf_group"].isna().sum())
print(f"SI/MA rows in ds for window {TIME_WINDOW}: {len(ds_sima)}")
print(f"  mapped to a waveform group: {len(ds_sima) - n_unmapped}")
print(f"  unmapped (no matching waveform unit): {n_unmapped}")
print(ds_sima["wf_group"].value_counts(dropna=False))

ds_glut = ds_sima[ds_sima["wf_group"] == "Glutamatergic"].copy()
ds_gaba = ds_sima[ds_sima["wf_group"] == "GABAergic"].copy()

## 8. Polar comparison: Glutamatergic vs. GABAergic

Same model (`reward` vs `chosenQ`, g0) and window as the reference call, but
drawn separately for each group.

In [ ]:
def polar_for_group(ds_group, label):
    """Draw the reward-vs-chosenQ polar plot for one waveform group."""
    if len(ds_group) == 0:
        print(f"[skip] {label}: no units.")
        return None
    fig, ax, tbl = plot_angle_fraction_polar(
        ds=ds_group,
        filter_region=TARGET_REGIONS,
        time_window=TIME_WINDOW,
        **MODEL_COLS,
        **POLAR_KW,
    )
    ax.set_title(f"{label} SI/MA (n={len(ds_group)})\nreward vs chosenQ (g0), {TIME_WINDOW}")
    plt.show()
    return fig, ax, tbl


res_glut = polar_for_group(ds_glut, "Glutamatergic")
res_gaba = polar_for_group(ds_gaba, "GABAergic")

### 8a. (Optional) Diagonal-significance view per group

In [ ]:
def diag_for_group(ds_group, label):
    """Draw the diagonal-significance scatter for one waveform group."""
    if len(ds_group) == 0:
        print(f"[skip] {label}: no units.")
        return None
    out = plot_diagonal_significance(
        ds=ds_group,
        filter_region=TARGET_REGIONS,
        time_window=TIME_WINDOW,
        **MODEL_COLS,
        point_size=1,
        fit_oval=False,
    )
    if isinstance(out, tuple) and len(out) >= 2:
        fig, ax = out[:2]
        ax.set_title(f"{label} SI/MA (n={len(ds_group)}) - reward vs chosenQ (g0), {TIME_WINDOW}")
    plt.show()
    return out


diag_for_group(ds_glut, "Glutamatergic")
diag_for_group(ds_gaba, "GABAergic")

## 9. Notes

- The two groups are defined **only by waveform morphology** on the SI/MA
  population; the opto-tagged units are used to decide *which* cluster is the
  Glutamatergic one, not to label units directly.
- Tune the split via `GROUP_FEATURE_COLS` (e.g. restrict to
  `["trough_to_peak_ms", "half_width_ms", "peak_trough_ratio"]` for a cleaner,
  more classic narrow-vs-broad separation).
- Unmapped `ds` rows are SI/MA correlation units that have no matching entry in
  the waveform-feature table (their session/unit is missing, failed QC, or was
  flagged as noise). They are excluded from both group plots.
- Adjust the opto-tagging thresholds / `TAG_TYPES` in the config cell to match
  how you defined tagged units elsewhere.